In [1]:
#imports
import pandas as pd
import numpy as np
from pathlib import Path

print(" Libraries loaded")
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

 Libraries loaded
Pandas version: 2.3.3
NumPy version: 2.3.5


In [2]:
# File paths for JupyterHub
from pathlib import Path

RAW_DIR = Path("churn_data/churn_data")
OUT_DIR = Path("my_output")

OUT_DIR.mkdir(exist_ok=True)

print(" Paths set")
print("Files found:")
for f in RAW_DIR.iterdir():
    print(" ", f.name)

 Paths set
Files found:
  customers.csv
  subscriptions.csv
  support_tickets.csv
  usage_monthly.csv


In [3]:
customers_raw = pd.read_csv(RAW_DIR / "customers.csv")
subs_raw      = pd.read_csv(RAW_DIR / "subscriptions.csv")
tickets_raw   = pd.read_csv(RAW_DIR / "support_tickets.csv")
usage_raw     = pd.read_csv(RAW_DIR / "usage_monthly.csv")

print(" All files loaded")

 All files loaded


In [4]:
# Row and column counts for all files
print("=" * 45)
print("FILE SIZES")
print("=" * 45)

files = {
    "customers"    : customers_raw,
    "subscriptions": subs_raw,
    "tickets"      : tickets_raw,
    "usage"        : usage_raw
}

for name, df in files.items():
    rows, cols = df.shape
    print(f"  {name:15s} → {rows:>6,} rows  |  {cols} columns")

FILE SIZES
  customers       →  3,030 rows  |  7 columns
  subscriptions   →  3,000 rows  |  7 columns
  tickets         →  4,618 rows  |  6 columns
  usage           → 39,712 rows  |  5 columns


In [5]:
# See column names and data types for each file
for name, df in files.items():
    print(f"\n{'='*45}")
    print(f"  {name.upper()}")
    print(f"{'='*45}")
    print(df.dtypes)


  CUSTOMERS
customer_id       object
signup_date       object
plan              object
region            object
acq_channel       object
contract_type     object
age              float64
dtype: object

  SUBSCRIPTIONS
customer_id      object
plan             object
mrr               int64
contract_type    object
signup_date      object
churn_date       object
status           object
dtype: object

  TICKETS
ticket_id       object
customer_id     object
created_date    object
category        object
priority        object
resolved        object
dtype: object

  USAGE
customer_id       object
month             object
logins             int64
features_used      int64
active_minutes     int64
dtype: object


In [6]:
# First 5 rows of each file
for name, df in files.items():
    print(f"\n{'='*45}")
    print(f"  {name.upper()} — first 5 rows")
    print(f"{'='*45}")
    display(df.head(5))


  CUSTOMERS — first 5 rows


,customer_id,signup_date,plan,region,acq_channel,contract_type,age
0,CUST01586,06-19-2023,Premium,East,Referral,Monthly,52.0
1,CUST01055,2024/08/04,Standard,East,Organic,Monthly,-5.0
2,CUST01784,04-20-2023,Basic,Central,Organic,Annual,30.0
3,CUST00072,2024-08-13,Basic,North,Social,Monthly,69.0
4,CUST02976,2025-01-01,Enterprise,East,Referral,Monthly,66.0



  SUBSCRIPTIONS — first 5 rows


,customer_id,plan,mrr,contract_type,signup_date,churn_date,status
0,CUST00001,Basic,299,Monthly,2024-10-16,2025-04-10,Churned
1,CUST00002,Premium,1199,Monthly,2024-11-23,NaN,Active
2,CUST00003,Basic,299,Monthly,2023-04-06,NaN,Active
3,CUST00004,Premium,1199,Monthly,2024-10-27,2025-03-16,Churned
4,CUST00005,Premium,1199,Monthly,2023-01-07,NaN,Active



  TICKETS — first 5 rows


,ticket_id,customer_id,created_date,category,priority,resolved
0,TKT001149,CUST00754,2025-03-15,Account,Low,Yes
1,TKT004097,CUST02649,2024-11-22,Other,Critical,No
2,TKT004575,CUST02987,2023-09-13,Billing,Low,Yes
3,TKT003387,CUST02214,2024-01-28,Other,Critical,Yes
4,TKT002570,CUST01685,2024-08-01,Technical,High,Yes



  USAGE — first 5 rows


,customer_id,month,logins,features_used,active_minutes
0,CUST00001,2024-10,0,14,62
1,CUST00001,2024-11,5,1,145
2,CUST00001,2024-12,0,11,581
3,CUST00001,2025-01,1,2,610
4,CUST00001,2025-02,0,12,668


In [7]:
# Null rate per column as percentage
print("=" * 45)
print("NULL RATES (%)")
print("=" * 45)

for name, df in files.items():
    null_rates = (df.isnull().sum() / len(df) * 100).round(2)
    has_nulls  = null_rates[null_rates > 0]
    
    print(f"\n  [{name}]")
    if len(has_nulls) == 0:
        print("    No nulls found ")
    else:
        for col, rate in has_nulls.items():
            severity = "HIGH" if rate > 20 else "🟡 MEDIUM" if rate > 5 else "🟢 LOW"
            print(f"    {col:20s} → {rate:5.1f}%  {severity}")
            

NULL RATES (%)

  [customers]
    region               →   3.7%  🟢 LOW
    age                  →   3.3%  🟢 LOW

  [subscriptions]
    churn_date           →  63.7%  HIGH

  [tickets]
    No nulls found 

  [usage]
    No nulls found 


In [8]:
# Check for duplicate rows in each file
print("=" * 45)
print("DUPLICATE ROWS")
print("=" * 45)

for name, df in files.items():
    # Full row duplicates
    full_dupes = df.duplicated().sum()
    print(f"\n  [{name}]")
    print(f"    Full duplicate rows   : {full_dupes}")

# Special checks — duplicate IDs (different from full row dupes)
print("\n  --- Duplicate KEY checks ---")

cust_id_dupes = customers_raw.duplicated(subset="customer_id").sum()
sub_id_dupes  = subs_raw.duplicated(subset="customer_id").sum()
tkt_id_dupes  = tickets_raw.duplicated(subset="ticket_id").sum()

print(f"  customers  → duplicate customer_id : {cust_id_dupes}")
print(f"  subscriptions → duplicate customer_id : {sub_id_dupes}")
print(f"  tickets    → duplicate ticket_id   : {tkt_id_dupes}")

DUPLICATE ROWS

  [customers]
    Full duplicate rows   : 30

  [subscriptions]
    Full duplicate rows   : 0

  [tickets]
    Full duplicate rows   : 0

  [usage]
    Full duplicate rows   : 0

  --- Duplicate KEY checks ---
  customers  → duplicate customer_id : 30
  subscriptions → duplicate customer_id : 0
  tickets    → duplicate ticket_id   : 0


In [9]:
# Anomalies in customers: age
print("AGE COLUMN ANOMALIES")
print("-" * 35)

print(f"  Min age : {customers_raw['age'].min()}")
print(f"  Max age : {customers_raw['age'].max()}")
print(f"  Mean age: {customers_raw['age'].mean():.1f}")

neg_age = customers_raw[customers_raw["age"] < 0]
print(f"\n  Negative age rows : {len(neg_age)}")
print(f"  Values found      : {sorted(neg_age['age'].unique().tolist())}")

old_age = customers_raw[customers_raw["age"] > 100]
print(f"\n  Age > 100 rows    : {len(old_age)}")

AGE COLUMN ANOMALIES
-----------------------------------
  Min age : -5.0
  Max age : 72.0
  Mean age: 44.2

  Negative age rows : 35
  Values found      : [-5.0]

  Age > 100 rows    : 0


In [10]:
# Mixed date formats in customers
print("DATE FORMAT ANOMALIES — customers.signup_date")
print("-" * 45)

sample_dates = customers_raw["signup_date"].dropna().head(20).tolist()
for d in sample_dates:
    print(f"  {d}")

DATE FORMAT ANOMALIES — customers.signup_date
---------------------------------------------
  06-19-2023
  2024/08/04
  04-20-2023
  2024-08-13
  2025-01-01
  06-07-2023
  12/05/2024
  2024/05/09
  2024/12/14
  23/01/2023
  08-13-2024
  03-22-2023
  30/08/2024
  2025-01-23
  04/02/2023
  01-11-2023
  2024-03-09
  11-29-2023
  2024-08-02
  2023-06-27


In [11]:
# Logical anomalies in subscriptions
print("SUBSCRIPTION LOGIC ANOMALIES")
print("-" * 45)

# Parse dates temporarily for checking
temp = subs_raw.copy()
temp["signup_date"] = pd.to_datetime(temp["signup_date"], errors="coerce")
temp["churn_date"]  = pd.to_datetime(temp["churn_date"],  errors="coerce")

# Problem 1: Active customer but has a churn_date
active_with_churn = temp[
    (temp["status"] == "Active") & temp["churn_date"].notna()
]
print(f"  Active status + churn_date populated : {len(active_with_churn)} rows ")

# Problem 2: Churned customer but churn_date is missing
churned_no_date = temp[
    (temp["status"] == "Churned") & temp["churn_date"].isna()
]
print(f"  Churned status + no churn_date       : {len(churned_no_date)} rows ")

# Problem 3: Churn date is BEFORE signup date (impossible)
bad_date_order = temp[temp["churn_date"] < temp["signup_date"]]
print(f"  churn_date before signup_date        : {len(bad_date_order)} rows ")

# Problem 4: Negative MRR
neg_mrr = subs_raw[subs_raw["mrr"] < 0]
print(f"  Negative MRR values                  : {len(neg_mrr)} rows")

SUBSCRIPTION LOGIC ANOMALIES
---------------------------------------------
  Active status + churn_date populated : 36 rows 
  Churned status + no churn_date       : 0 rows 
  churn_date before signup_date        : 60 rows 
  Negative MRR values                  : 0 rows


In [12]:
# Orphan records (tickets/usage with no matching customer)
print("REFERENTIAL INTEGRITY CHECKS")
print("-" * 45)

valid_customer_ids = set(customers_raw["customer_id"])

# Tickets with no matching customer
orphan_tickets = tickets_raw[~tickets_raw["customer_id"].isin(valid_customer_ids)]
print(f"  Orphan tickets (no matching customer) : {len(orphan_tickets)} rows ")
print(f"  Sample IDs: {orphan_tickets['customer_id'].unique()[:5].tolist()}")

# Usage with no matching customer
orphan_usage = usage_raw[~usage_raw["customer_id"].isin(valid_customer_ids)]
print(f"\n  Orphan usage rows (no matching customer): {len(orphan_usage)} rows")

REFERENTIAL INTEGRITY CHECKS
---------------------------------------------
  Orphan tickets (no matching customer) : 20 rows 
  Sample IDs: ['CUST90011', 'CUST98430', 'CUST99560', 'CUST90638', 'CUST92492']

  Orphan usage rows (no matching customer): 0 rows


In [13]:
# Negative values in usage
print("USAGE COLUMN SANITY CHECK")
print("-" * 45)

for col in ["logins", "features_used", "active_minutes"]:
    neg_count = (usage_raw[col] < 0).sum()
    zero_count = (usage_raw[col] == 0).sum()
    print(f"  {col:20s} → negatives: {neg_count}  |  zeros: {zero_count}")

USAGE COLUMN SANITY CHECK
---------------------------------------------
  logins               → negatives: 0  |  zeros: 4737
  features_used        → negatives: 0  |  zeros: 0
  active_minutes       → negatives: 0  |  zeros: 0


In [14]:
print("\nDATA QUALITY FINDINGS SUMMARY")
print("-" * 50)

print("1. customers      - duplicate rows        - HIGH")
print("2. customers      - negative ages         - MEDIUM")
print("3. customers      - mixed date formats    - MEDIUM")
print("4. customers      - null regions          - LOW")
print("5. subscriptions  - active + churn date   - HIGH")
print("6. subscriptions  - churn before signup   - HIGH")
print("7. tickets        - orphan records        - LOW")


DATA QUALITY FINDINGS SUMMARY
--------------------------------------------------
1. customers      - duplicate rows        - HIGH
2. customers      - negative ages         - MEDIUM
3. customers      - mixed date formats    - MEDIUM
4. customers      - null regions          - LOW
5. subscriptions  - active + churn date   - HIGH
6. subscriptions  - churn before signup   - HIGH
7. tickets        - orphan records        - LOW


In [15]:
# Save findings to a JSON file
import json
from pathlib import Path

OUT_DIR = Path(r"C:\Users\raksh\Downloads\churn_assignment_rakshit_kaushal\submission\my_output")
OUT_DIR.mkdir(exist_ok=True)

quality_report = {
    "customers": {
        "total_rows"       : len(customers_raw),
        "duplicate_rows"   : int(customers_raw.duplicated().sum()),
        "null_region_%"    : round(customers_raw["region"].isnull().mean()*100, 2),
        "null_age_%"       : round(customers_raw["age"].isnull().mean()*100, 2),
        "negative_age_rows": int((customers_raw["age"] < 0).sum()),
        "mixed_date_formats": True
    },
    "subscriptions": {
        "total_rows"              : len(subs_raw),
        "active_with_churn_date"  : int(((subs_raw["status"]=="Active") & subs_raw["churn_date"].notna()).sum()),
        "churn_before_signup"     : 60,
        "null_churn_date_%"       : round(subs_raw["churn_date"].isnull().mean()*100, 2)
    },
    "tickets": {
        "total_rows"    : len(tickets_raw),
        "orphan_records": int((~tickets_raw["customer_id"].isin(customers_raw["customer_id"])).sum())
    },
    "usage": {
        "total_rows": len(usage_raw)
    }
}

with open(OUT_DIR / "quality_report.json", "w") as f:
    json.dump(quality_report, f, indent=2)

print(" Quality report saved → my_output/quality_report.json")

 Quality report saved → my_output/quality_report.json


In [16]:

cust  = customers_raw.copy()
subs  = subs_raw.copy()
tix   = tickets_raw.copy()
usage = usage_raw.copy()

print(" Working copies created")
print("  cust  :", cust.shape)
print("  subs  :", subs.shape)
print("  tix   :", tix.shape)
print("  usage :", usage.shape)

 Working copies created
  cust  : (3030, 7)
  subs  : (3000, 7)
  tix   : (4618, 6)
  usage : (39712, 5)


In [17]:


before = len(cust)

cust = cust.drop_duplicates(subset="customer_id", keep="first")

after = len(cust)

print(f"  Rows before : {before}")
print(f"  Rows after  : {after}")
print(f"  Removed     : {before - after} duplicate rows ")

  Rows before : 3030
  Rows after  : 3000
  Removed     : 30 duplicate rows 


In [18]:

print("Before cleaning — sample dates:")
print(cust["signup_date"].head(5).tolist())

cust["signup_date"] = pd.to_datetime(
    cust["signup_date"],
    format="mixed",      # handles multiple formats automatically
    dayfirst=False,
    errors="coerce"      # if it can't parse, set to null instead of crashing
)

print("\nAfter cleaning — sample dates:")
print(cust["signup_date"].head(5).tolist())
print(f"\nNull dates after parsing: {cust['signup_date'].isnull().sum()}")
print("Date formats standardised")

Before cleaning — sample dates:
['06-19-2023', '2024/08/04', '04-20-2023', '2024-08-13', '2025-01-01']

After cleaning — sample dates:
[Timestamp('2023-06-19 00:00:00'), Timestamp('2024-08-04 00:00:00'), Timestamp('2023-04-20 00:00:00'), Timestamp('2024-08-13 00:00:00'), Timestamp('2025-01-01 00:00:00')]

Null dates after parsing: 0
Date formats standardised


In [19]:
#  Handle negative age values
# Problem: 35 rows have age = -5 (impossible, likely system default)
# Decision: Flag them, then null them out
# Why not delete: we keep the customer row, just lose the age value

# Step 1: Add a flag column (empty string for all rows first)
cust["age_flag"] = ""

# Step 2: Flag the bad rows
cust.loc[cust["age"] < 0, "age_flag"] = "NEGATIVE_AGE"

# Step 3: Null out the impossible values
cust.loc[cust["age"] < 0, "age"] = np.nan

# Check result
print("Age flag counts:")
print(cust["age_flag"].value_counts())
print(f"\nNegative ages remaining: {(cust['age'] < 0).sum()}")
print("Negative ages flagged and nulled")

Age flag counts:
age_flag
                2966
NEGATIVE_AGE      34
Name: count, dtype: int64

Negative ages remaining: 0
Negative ages flagged and nulled


In [20]:

cust["plan"]          = cust["plan"].str.strip().str.title()
cust["region"]        = cust["region"].str.strip().str.title()
cust["acq_channel"]   = cust["acq_channel"].str.strip().str.title()
cust["contract_type"] = cust["contract_type"].str.strip().str.title()

print("Unique plan values:")
print(cust["plan"].unique())

print("\nUnique region values:")
print(cust["region"].unique())

print("\nUnique contract_type values:")
print(cust["contract_type"].unique())

print("\n Text columns standardised")

Unique plan values:
['Premium' 'Standard' 'Basic' 'Enterprise']

Unique region values:
['East' 'Central' 'North' 'South' 'West' nan]

Unique contract_type values:
['Monthly' 'Annual']

 Text columns standardised


In [21]:
# Parse dates in subscriptions
subs["signup_date"] = pd.to_datetime(subs["signup_date"], errors="coerce")
subs["churn_date"]  = pd.to_datetime(subs["churn_date"],  errors="coerce")

# Standardise text
subs["plan"]          = subs["plan"].str.strip().str.title()
subs["contract_type"] = subs["contract_type"].str.strip().str.title()
subs["status"]        = subs["status"].str.strip().str.title()

print("Unique status values:", subs["status"].unique())
print("Unique plan values  :", subs["plan"].unique())
print(" Subscriptions dates and text cleaned")

Unique status values: ['Churned' 'Active']
Unique plan values  : ['Basic' 'Premium' 'Enterprise' 'Standard']
 Subscriptions dates and text cleaned


In [22]:
# Cell 22 — Fix Active customers who have a churn_date
# Problem: 36 rows say status=Active but have a churn_date filled in
# This is a contradiction — active customers should not have a churn date
# Decision: null out the churn_date, flag the row
# Why: trust the status column more than the date

subs["sub_flag"] = ""

# Find the problem rows
mask_active_churn = (subs["status"] == "Active") & (subs["churn_date"].notna())

# Flag them
subs.loc[mask_active_churn, "sub_flag"] = "ACTIVE_WITH_CHURN_DATE"

# Null out the churn_date
subs.loc[mask_active_churn, "churn_date"] = pd.NaT

print(f"  Rows fixed : {mask_active_churn.sum()}")
print(f"  Flag counts:")
print(subs["sub_flag"].value_counts())
print(" Active + churn_date conflict resolved")

  Rows fixed : 38
  Flag counts:
sub_flag
                          2962
ACTIVE_WITH_CHURN_DATE      38
Name: count, dtype: int64
 Active + churn_date conflict resolved


In [23]:
# Fix churn_date before signup_date
# Problem: 60 rows where churn happened BEFORE the customer signed up
# This is physically impossible
# Decision: flag and null the churn_date
# Why: we cannot know the correct churn_date so we remove it

mask_bad_order = (
    subs["churn_date"].notna() &
    (subs["churn_date"] < subs["signup_date"])
)

# Flag them
subs.loc[mask_bad_order, "sub_flag"] = "CHURN_BEFORE_SIGNUP"

# Null out impossible churn_date
subs.loc[mask_bad_order, "churn_date"] = pd.NaT

print(f"  Rows fixed : {mask_bad_order.sum()}")
print(" Impossible churn dates flagged and nulled")

  Rows fixed : 22
 Impossible churn dates flagged and nulled


In [24]:
# Cell 24 — Clean support tickets
tix["created_date"] = pd.to_datetime(tix["created_date"], errors="coerce")
tix["priority"]     = tix["priority"].str.strip().str.title()
tix["category"]     = tix["category"].str.strip().str.title()
tix["resolved"]     = tix["resolved"].str.strip().str.title()

# Flag orphan tickets
tix["orphan_flag"] = ~tix["customer_id"].isin(cust["customer_id"])

print("Unique priority values:", tix["priority"].unique())
print("Unique resolved values :", tix["resolved"].unique())
print(f"Orphan tickets flagged : {tix['orphan_flag'].sum()}")
print(" Tickets cleaned")

Unique priority values: ['Low' 'Critical' 'High' 'Medium']
Unique resolved values : ['Yes' 'No']
Orphan tickets flagged : 20
 Tickets cleaned


In [25]:
# Clean usage data
usage["month"] = pd.to_datetime(usage["month"], errors="coerce")

# Clip any negative values to 0 (physically impossible)
for col in ["logins", "features_used", "active_minutes"]:
    neg_count = (usage[col] < 0).sum()
    if neg_count > 0:
        print(f"  Clipping {neg_count} negative values in {col}")
    usage[col] = usage[col].clip(lower=0)

# Flag orphan usage rows
usage["orphan_flag"] = ~usage["customer_id"].isin(cust["customer_id"])

print(f"Orphan usage rows: {usage['orphan_flag'].sum()}")
print(" Usage cleaned")

Orphan usage rows: 0
 Usage cleaned


In [26]:
# Final check after all cleaning
print("=" * 45)
print("CLEANED ROW COUNTS")
print("=" * 45)
print(f"  customers     : {len(cust)}")
print(f"  subscriptions : {len(subs)}")
print(f"  tickets       : {len(tix)}")
print(f"  usage         : {len(usage)}")

print("\n" + "=" * 45)
print("REMAINING NULLS")
print("=" * 45)
for name, df in [("cust", cust), ("subs", subs)]:
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    print(f"\n  [{name}]")
    for col, n in nulls.items():
        print(f"    {col:25s}: {n}")

print("\n A2 Cleaning Complete")

CLEANED ROW COUNTS
  customers     : 3000
  subscriptions : 3000
  tickets       : 4618
  usage         : 39712

REMAINING NULLS

  [cust]
    region                   : 111
    age                      : 133

  [subs]
    churn_date               : 1972

 A2 Cleaning Complete


In [27]:
# Save cleaned dimension tables
OUT_DIR = Path(r"C:\Users\raksh\Downloads\churn_assignment_rakshit_kaushal\submission\my_output")
OUT_DIR.mkdir(exist_ok=True)

cust.to_csv(OUT_DIR  / "dim_customers.csv",         index=False)
subs.to_csv(OUT_DIR  / "dim_subscriptions.csv",     index=False)
tix.to_csv(OUT_DIR   / "fact_support_tickets.csv",  index=False)
usage.to_csv(OUT_DIR / "fact_usage_monthly.csv",    index=False)

print(" All clean tables saved to my_output/")
print("  dim_customers.csv")
print("  dim_subscriptions.csv")
print("  fact_support_tickets.csv")
print("  fact_usage_monthly.csv")

 All clean tables saved to my_output/
  dim_customers.csv
  dim_subscriptions.csv
  fact_support_tickets.csv
  fact_usage_monthly.csv


In [28]:

ANALYSIS_DATE = pd.Timestamp("2025-06-01")  # our "today"

usage_clean = usage[~usage["orphan_flag"]]  # exclude orphan rows

usage_agg = (
    usage_clean
    .groupby("customer_id")
    .agg(
        avg_monthly_logins    = ("logins",         "mean"),
        avg_features_used     = ("features_used",  "mean"),
        avg_active_minutes    = ("active_minutes",  "mean"),
        total_months_with_data= ("month",           "count"),
        last_active_month     = ("month",           "max")
    )
    .reset_index()
)

# Round to 2 decimal places
usage_agg["avg_monthly_logins"]  = usage_agg["avg_monthly_logins"].round(2)
usage_agg["avg_features_used"]   = usage_agg["avg_features_used"].round(2)
usage_agg["avg_active_minutes"]  = usage_agg["avg_active_minutes"].round(2)

print(f" Usage aggregated: {usage_agg.shape}")
display(usage_agg.head(3))

 Usage aggregated: (3000, 6)


,customer_id,avg_monthly_logins,avg_features_used,avg_active_minutes,total_months_with_data,last_active_month
0,CUST00001,0.86,7.43,583.14,7,2025-04-01
1,CUST00002,5.67,9.17,359.17,6,2025-04-01
2,CUST00003,12.44,7.64,624.40,25,2025-04-01


In [29]:
# Logins in last 90 days (recency signal)
# Why: a customer active last month is very different from 
# one who hasn't logged in for 3 months

cutoff_90 = ANALYSIS_DATE - pd.Timedelta(days=90)

usage_recent = (
    usage_clean[usage_clean["month"] >= cutoff_90]
    .groupby("customer_id")
    .agg(logins_last90d = ("logins", "sum"))
    .reset_index()
)

print(f" Recent usage aggregated: {usage_recent.shape}")
display(usage_recent.head(3))

 Recent usage aggregated: (2118, 2)


,customer_id,logins_last90d
0,CUST00001,0
1,CUST00002,9
2,CUST00003,13


In [30]:
# Summarise tickets from one row per ticket - one row per customer

tix_clean = tix[~tix["orphan_flag"]]  # exclude orphan tickets

ticket_agg = (
    tix_clean
    .groupby("customer_id")
    .agg(
        total_tickets      = ("ticket_id", "count"),
        unresolved_tickets = ("resolved",  lambda x: (x == "No").sum()),
        critical_tickets   = ("priority",  lambda x: (x == "Critical").sum())
    )
    .reset_index()
)

print(f" Tickets aggregated: {ticket_agg.shape}")
display(ticket_agg.head(3))

 Tickets aggregated: (1920, 4)


,customer_id,total_tickets,unresolved_tickets,critical_tickets
0,CUST00003,2,0,0
1,CUST00004,1,0,0
2,CUST00007,3,0,0


In [31]:
# Ticket count in last 90 days
# Why: recent support issues are a stronger churn signal
# than tickets raised 2 years ago

cutoff_90t = ANALYSIS_DATE - pd.Timedelta(days=90)

tickets_recent = (
    tix_clean[tix_clean["created_date"] >= cutoff_90t]
    .groupby("customer_id")
    .size()
    .reset_index(name="tickets_last90d")
)

print(f" Recent tickets aggregated: {tickets_recent.shape}")
display(tickets_recent.head(3))

 Recent tickets aggregated: (855, 2)


,customer_id,tickets_last90d
0,CUST00003,1
1,CUST00010,1
2,CUST00012,2


In [32]:
# joining all 4 tables into one flat table
df = (
    cust
    .merge(
        subs[["customer_id","plan","mrr","contract_type",
              "signup_date","churn_date","status","sub_flag"]],
        on="customer_id",
        how="left",
        suffixes=("_cust", "_sub")
    )
    .merge(usage_agg,      on="customer_id", how="left")
    .merge(usage_recent,   on="customer_id", how="left")
    .merge(ticket_agg,     on="customer_id", how="left")
    .merge(tickets_recent, on="customer_id", how="left")
)

print(f" Master table built: {df.shape}")
print(f"   Rows    : {df.shape[0]:,}")
print(f"   Columns : {df.shape[1]}")
display(df.head(3))

 Master table built: (3000, 25)
   Rows    : 3,000
   Columns : 25


,customer_id,signup_date_cust,plan_cust,region,acq_channel,contract_type_cust,age,age_flag,plan_sub,mrr,...,avg_monthly_logins,avg_features_used,avg_active_minutes,total_months_with_data,last_active_month,logins_last90d,total_tickets,unresolved_tickets,critical_tickets,tickets_last90d
0,CUST01586,2023-06-19,Premium,East,Referral,Monthly,52.0,,Premium,1199,...,11.22,8.17,564.61,23,2025-04-01,13.0,6.0,2.0,0.0,1.0
1,CUST01055,2024-08-04,Standard,East,Organic,Monthly,NaN,NEGATIVE_AGE,Standard,599,...,1.78,9.56,363.56,9,2025-04-01,2.0,NaN,NaN,NaN,NaN
2,CUST01784,2023-04-20,Basic,Central,Organic,Annual,30.0,,Basic,299,...,7.64,7.44,564.80,25,2025-04-01,9.0,NaN,NaN,NaN,NaN


In [33]:
# Engineer new features needed for churn analysis

# 1. Resolve plan conflict (subscription is authoritative)
df["plan_final"] = df["plan_sub"].combine_first(df["plan_cust"])
df["contract_type_final"] = df["contract_type_sub"].combine_first(df["contract_type_cust"])

# 2. Churn flag (1 = churned, 0 = active)
df["churned"] = (df["status"] == "Churned").astype(int)

# 3. Tenure in days
signup    = df["signup_date_sub"].combine_first(df["signup_date_cust"])
end_date  = df["churn_date"].where(df["churned"] == 1, other=ANALYSIS_DATE)
df["tenure_days"] = (end_date - signup).dt.days.clip(lower=0)

# 4. Tenure bucket
bins   = [0, 90, 180, 365, 730, 9999]
labels = ["0-3m", "3-6m", "6-12m", "12-24m", "24m+"]
df["tenure_bucket"] = pd.cut(
    df["tenure_days"],
    bins=bins,
    labels=labels,
    right=False
)

# 5. Engagement tier based on avg monthly logins
df["engagement_tier"] = pd.cut(
    df["avg_monthly_logins"].fillna(0),
    bins=[-1, 0, 5, 15, 9999],
    labels=["Inactive", "Low", "Medium", "High"]
)

# 6. Days since last active
df["days_since_last_active"] = (
    ANALYSIS_DATE - df["last_active_month"]
).dt.days

print(" Features engineered")
print("\nNew columns added:")
new_cols = ["plan_final","contract_type_final","churned",
            "tenure_days","tenure_bucket","engagement_tier",
            "days_since_last_active"]
for col in new_cols:
    print(f"  {col}")

 Features engineered

New columns added:
  plan_final
  contract_type_final
  churned
  tenure_days
  tenure_bucket
  engagement_tier
  days_since_last_active


In [34]:
# Fill nulls
# Customers with no tickets → 0 tickets (not missing, genuinely zero)
# Customers with no usage → 0 logins (they never logged in)

zero_fill_cols = [
    "total_tickets", "unresolved_tickets", "critical_tickets",
    "tickets_last90d", "logins_last90d",
    "avg_monthly_logins", "avg_features_used",
    "avg_active_minutes", "total_months_with_data"
]

df[zero_fill_cols] = df[zero_fill_cols].fillna(0)

print("Nulls filled with 0 for ticket and usage columns")
print(f"\nFinal table: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Churned customers : {df['churned'].sum():,}")
print(f"Active customers  : {(df['churned']==0).sum():,}")
print(f"Overall churn rate: {df['churned'].mean()*100:.1f}%")

Nulls filled with 0 for ticket and usage columns

Final table: 3,000 rows × 32 columns
Churned customers : 1,050
Active customers  : 1,950
Overall churn rate: 35.0%


In [35]:
# Save the master analytical table
# file for Tableau

df.to_csv(OUT_DIR / "analytical_customer_features.csv", index=False)

print(" Master analytical table saved")
print(f"   → my_output/analytical_customer_features.csv")
print(f"   {df.shape[0]:,} rows × {df.shape[1]} columns")
print("\nColumns in final table:")
for col in df.columns:
    print(f"  {col}")

 Master analytical table saved
   → my_output/analytical_customer_features.csv
   3,000 rows × 32 columns

Columns in final table:
  customer_id
  signup_date_cust
  plan_cust
  region
  acq_channel
  contract_type_cust
  age
  age_flag
  plan_sub
  mrr
  contract_type_sub
  signup_date_sub
  churn_date
  status
  sub_flag
  avg_monthly_logins
  avg_features_used
  avg_active_minutes
  total_months_with_data
  last_active_month
  logins_last90d
  total_tickets
  unresolved_tickets
  critical_tickets
  tickets_last90d
  plan_final
  contract_type_final
  churned
  tenure_days
  tenure_bucket
  engagement_tier
  days_since_last_active
